<a href="https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliMahmoud67/FlyRank_Starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

## Research Question

Can observable search and content signals be used to rank pages that are worth reviewing or refreshing, so editors can prioritize their limited content-review time?

### Decision supported

The output is a ranked list of content items that should be reviewed first.

### Unit of analysis

One row represents one content item for one client.

### Output

A priority score, rank, action label, and reason code.

### Human action

A FlyRank editor can use the ranked queue to decide which pages to review or refresh first.

### Cost of a wrong decision

A false positive sends editor time toward a page that may not need attention. A false negative may cause a page that could benefit from review to be missed.

### Why data and ML help

Content performance depends on several signals at the same time, such as search visibility, click-through performance, and content age. A data-driven approach can help prioritize these signals consistently instead of relying only on manual review.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data

This project uses the FlyRank internship warehouse release hosted on Hugging Face.

### Tables used

- `fact_content_daily_performance` — daily search performance for each client and content item.
- `dim_content` — content-level information such as word count and content creation date.

### Time windows

- **Feature window:** February 2026
- **Outcome window:** March 2026

February data is used for the features and March data is used only as the future outcome for evaluation.

### Main fields used

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `word_count`
- `content_age_days`

### Excluded fields

`trend_direction`, `trend_pct`, and `is_declining_label` were excluded because they are derived from trend information and can leak the outcome.

Pseudonymous client and content IDs are used only for grouping and joining. They are not model features.

### Data limitation

GSC data is not available for every row. Missing GSC data is treated as unavailable data rather than zero performance.

In [12]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

print("Connected to FlyRank warehouse.")

Connected to FlyRank warehouse.


In [13]:
FEB = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet"
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

feb_check = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{FEB}')
""").df()

mar_check = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{MAR}')
""").df()

print("February 2026:")
print(feb_check)

print("\nMarch 2026:")
print(mar_check)

February 2026:
      rows start_date   end_date
0  7355108 2026-02-01 2026-02-28

March 2026:
      rows start_date   end_date
0  9841378 2026-03-01 2026-03-31


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*


## Methodology

This project treats the task as a ranking problem: the model should assign each content item a score representing how useful it may be to review or refresh.

### Features

The model uses five February features:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `word_count`
- `content_age_days`

All five features are available before the March outcome window.

### Target / proxy

The target is a future content-opportunity proxy. A page is marked as a future opportunity when its March CTR is in the bottom 25% among pages with a similar February search position, using pages with at least 100 March impressions.

This is a proxy for identifying pages that receive relatively weak click performance despite having comparable search visibility.

### Baseline

The baseline is a simple transparent score:

- +1 for stale content (`content_age_days >= 180`)
- +1 for weak February CTR relative to pages with a similar search position

The resulting score is 0, 1, or 2 and is used to create the baseline ranked queue.

### Validation design

The final evaluation will use a time-aware split: earlier month pairs will be used for training and the February-to-March period will be held out as the future test period.

The model and baseline will be evaluated on the same test population and with the same ranking metric.

### Leakage checks

No March performance fields are used as model features.

The model also excludes `trend_direction`, `trend_pct`, `is_declining_label`, product decision flags, and pseudonymous IDs from the feature set.

The feature window is strictly before the outcome window.

In [14]:
import pandas as pd
import numpy as np

DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# ---------------------------------------------------------
# February features
# ---------------------------------------------------------

feature_vector = con.execute(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(f.gsc_impressions) AS gsc_impressions,
        SUM(f.gsc_clicks) AS gsc_clicks,

        SUM(f.gsc_sum_position)
            / NULLIF(SUM(f.gsc_impressions), 0) AS gsc_avg_position,

        d.word_count,

        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-02-28'
        ) AS content_age_days

    FROM read_parquet('{FEB}') f

    JOIN read_parquet('{DIM}') d
        ON f.content_hash_id = d.content_hash_id

    WHERE f.gsc_data_available IS TRUE
      AND d.content_created_date <= DATE '2026-02-28'

    GROUP BY
        f.client_hash_id,
        f.content_hash_id,
        d.word_count,
        d.content_created_date
""").df()

# ---------------------------------------------------------
# March outcome
# ---------------------------------------------------------

march = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,

        SUM(gsc_clicks)
            / NULLIF(SUM(gsc_impressions), 0) AS march_ctr

    FROM read_parquet('{MAR}')

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 100
""").df()

# Combine February features with the future March outcome
capstone_df = feature_vector.merge(
    march,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Only pages with a useful February search position
capstone_df = capstone_df[
    (capstone_df["gsc_avg_position"] > 0) &
    (capstone_df["gsc_avg_position"] <= 20)
].copy()

# Position groups
capstone_df["position_group"] = pd.cut(
    capstone_df["gsc_avg_position"],
    [0, 3, 10, 20],
    labels=["1-3", "4-10", "11-20"]
)

# ---------------------------------------------------------
# Future target
# ---------------------------------------------------------

capstone_df["future_ctr_rank"] = (
    capstone_df
    .groupby("position_group", observed=True)["march_ctr"]
    .rank(method="min", pct=True)
)

capstone_df["future_opportunity"] = (
    capstone_df["future_ctr_rank"] <= 0.25
)

print("Evaluation rows:", len(capstone_df))
print("Future opportunity rate:",
      round(capstone_df["future_opportunity"].mean(), 4))

print("\nTarget counts:")
print(capstone_df["future_opportunity"].value_counts())

capstone_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "word_count",
        "content_age_days",
        "march_ctr",
        "future_opportunity"
    ]
].head(10)

Evaluation rows: 73430
Future opportunity rate: 0.3348

Target counts:
future_opportunity
False    48843
True     24587
Name: count, dtype: int64


,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days,march_ctr,future_opportunity
0,1598.0,7.0,17.273467,2689,226,0.001954,False
1,245.0,0.0,8.032653,3108,226,0.000000,True
2,970.0,2.0,10.307216,2396,226,0.000996,False
3,688.0,0.0,13.879360,2949,226,0.000000,True
4,291.0,0.0,3.972509,2781,226,0.002907,False
5,4399.0,1.0,0.949079,3018,226,0.000000,True
7,3641.0,6.0,8.829992,2685,226,0.004276,False
8,817.0,0.0,12.891065,2667,226,0.000863,False
9,753.0,0.0,15.827357,2681,226,0.000000,True
10,932.0,0.0,15.736052,2759,226,0.004456,False


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## Model

We use a Random Forest classifier to estimate which content items are future review opportunities.

The model predicts the binary `future_opportunity` target using the five February features:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `word_count`
- `content_age_days`

The model produces a probability score for each page. Pages are then ranked by that score.

The model does not use March outcome fields, trend-derived labels, product flags, or pseudonymous IDs as features.

In [15]:
# Build historical training data
# We use older month -> next month pairs for training.
# February -> March stays as our final test period.

from sklearn.ensemble import RandomForestClassifier

train_months = [
    ("2025-08", "2025-09"),
    ("2025-09", "2025-10"),
    ("2025-10", "2025-11"),
    ("2025-11", "2025-12"),
    ("2025-12", "2026-01"),
    ("2026-01", "2026-02"),
]

def build_month_pair(feature_month, outcome_month):

    feature_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={feature_month}/*.parquet"
    )

    outcome_path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/month={outcome_month}/*.parquet"
    )

    # Features from the earlier month
    features = con.execute(f"""
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(f.gsc_impressions) AS gsc_impressions,
            SUM(f.gsc_clicks) AS gsc_clicks,

            SUM(f.gsc_sum_position)
                / NULLIF(SUM(f.gsc_impressions), 0) AS gsc_avg_position,

            d.word_count,

            DATE_DIFF(
                'day',
                d.content_created_date,
                LAST_DAY(DATE '{feature_month}-01')
            ) AS content_age_days

        FROM read_parquet('{feature_path}') f

        JOIN read_parquet('{DIM}') d
            ON f.content_hash_id = d.content_hash_id

        WHERE f.gsc_data_available IS TRUE
          AND d.content_created_date <= LAST_DAY(DATE '{feature_month}-01')

        GROUP BY
            f.client_hash_id,
            f.content_hash_id,
            d.word_count,
            d.content_created_date
    """).df()

    # Future CTR
    outcomes = con.execute(f"""
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS future_impressions,
            SUM(gsc_clicks)
                / NULLIF(SUM(gsc_impressions), 0) AS future_ctr

        FROM read_parquet('{outcome_path}')

        WHERE gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING SUM(gsc_impressions) >= 100
    """).df()

    df = features.merge(
        outcomes,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )

    # Keep pages with a useful search position
    df = df[
        (df["gsc_avg_position"] > 0) &
        (df["gsc_avg_position"] <= 20)
    ].copy()

    # Compare pages with similar search positions
    df["position_group"] = pd.cut(
        df["gsc_avg_position"],
        [0, 3, 10, 20],
        labels=["1-3", "4-10", "11-20"]
    )

    # Future opportunity = bottom 25% future CTR in that position group
    df["future_ctr_rank"] = (
        df.groupby("position_group", observed=True)["future_ctr"]
        .rank(method="min", pct=True)
    )

    df["future_opportunity"] = (
        df["future_ctr_rank"] <= 0.25
    )

    return df


training_parts = []

for feature_month, outcome_month in train_months:
    print(f"Building {feature_month} -> {outcome_month}...")
    part = build_month_pair(feature_month, outcome_month)
    print("Rows:", len(part))
    training_parts.append(part)

train_df = pd.concat(
    training_parts,
    ignore_index=True
)

print("\nTotal training rows:", len(train_df))
print(
    "Training opportunity rate:",
    round(train_df["future_opportunity"].mean(), 4)
)

Building 2025-08 -> 2025-09...
Rows: 11322
Building 2025-09 -> 2025-10...
Rows: 21661
Building 2025-10 -> 2025-11...
Rows: 29605
Building 2025-11 -> 2025-12...
Rows: 43172
Building 2025-12 -> 2026-01...
Rows: 49203
Building 2026-01 -> 2026-02...
Rows: 57352

Total training rows: 212315
Training opportunity rate: 0.3108


In [16]:
# February -> March is our final test period

test_df = capstone_df.copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "\nTraining opportunity rate:",
    round(train_df["future_opportunity"].mean(), 4)
)

print(
    "Test opportunity rate:",
    round(test_df["future_opportunity"].mean(), 4)
)

Training rows: 212315
Test rows: 73430

Training opportunity rate: 0.3108
Test opportunity rate: 0.3348


In [17]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

# Features used by the model
FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "content_age_days"
]

# Remove rows with missing feature values
train_model = train_df.dropna(subset=FEATURES + ["future_opportunity"]).copy()
test_model = test_df.dropna(subset=FEATURES + ["future_opportunity"]).copy()

X_train = train_model[FEATURES]
y_train = train_model["future_opportunity"]

X_test = test_model[FEATURES]
y_test = test_model["future_opportunity"]

# Train the model
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=10
)

model.fit(X_train, y_train)

# Model probability = ranking score
test_model["model_score"] = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Baseline score
# ---------------------------------------------------------

test_model["feb_ctr"] = (
    test_model["gsc_clicks"] /
    test_model["gsc_impressions"].replace(0, np.nan)
)

test_model["stale"] = (
    test_model["content_age_days"] >= 180
)

test_model["position_group"] = pd.cut(
    test_model["gsc_avg_position"],
    [0, 3, 10, 20],
    labels=["1-3", "4-10", "11-20"]
)

test_model["ctr_rank"] = (
    test_model
    .groupby("position_group", observed=True)["feb_ctr"]
    .rank(method="min", pct=True)
)

test_model["weak_ctr"] = test_model["ctr_rank"] <= 0.25

test_model["baseline_score"] = (
    test_model["stale"].astype(int) +
    test_model["weak_ctr"].fillna(False).astype(int)
)

# ---------------------------------------------------------
# Precision@K
# ---------------------------------------------------------

def precision_at_k(df, score_column, k):
    ranked = df.sort_values(score_column, ascending=False).head(k)
    return ranked["future_opportunity"].mean()

for k in [20, 50]:
    model_precision = precision_at_k(
        test_model,
        "model_score",
        k
    )

    baseline_precision = precision_at_k(
        test_model,
        "baseline_score",
        k
    )

    print(f"\nPrecision@{k}")
    print(f"Baseline: {baseline_precision:.4f}")
    print(f"Model:    {model_precision:.4f}")

print("\nTest rows:", len(test_model))
print("Base rate:", round(test_model["future_opportunity"].mean(), 4))


Precision@20
Baseline: 0.6500
Model:    0.7500

Precision@50
Baseline: 0.6600
Model:    0.7800

Test rows: 50625
Base rate: 0.2704


In [18]:
# Feature importance

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

            feature  importance
1        gsc_clicks    0.334695
0   gsc_impressions    0.287160
3        word_count    0.148093
4  content_age_days    0.115977
2  gsc_avg_position    0.114076


In [19]:
# Error analysis: inspect the model's top 20 predictions

top20 = (
    test_model
    .sort_values("model_score", ascending=False)
    .head(20)
    .copy()
)

top20["correct"] = (
    top20["future_opportunity"] == True
)

print("Top 20 model-ranked pages:")
print(
    top20[
        [
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "future_opportunity",
            "correct"
        ]
    ]
)

print(
    "\nWrong picks in top 20:",
    (~top20["correct"]).sum()
)

Top 20 model-ranked pages:
                client_hash_id           content_hash_id  model_score  \
6180   client_fef1a8f436438636  content_a0caba51499ad66e     0.919694   
5767   client_fef1a8f436438636  content_9ee4f0a3b3f41d9a     0.919598   
54440  client_23a62021009f63c4  content_cf445d75a938ea9a     0.913005   
7982   client_e547b89c05043229  content_e70d9d75f6b54ddb     0.908520   
16326  client_fef1a8f436438636  content_649f6f28999bdf02     0.905359   
52715  client_400c21c81c8b46ef  content_685ba8c22f4666d5     0.903562   
6050   client_fef1a8f436438636  content_e88c4faaa3394802     0.900785   
54288  client_23a62021009f63c4  content_d932a8c55a6ce23b     0.900116   
5754   client_fef1a8f436438636  content_6f1837650aafcf4c     0.899100   
11971  client_9958f0a7ae1df715  content_fbb1339100bd5e5b     0.897817   
11945  client_9958f0a7ae1df715  content_3098d853d241f6cf     0.895256   
30482  client_3197e6291363b4db  content_fa3d03c4de7ca6c6     0.894839   
52483  client_400c21c81c

In [20]:
# Inspect the wrong predictions in the top 20

wrong_top20 = top20[
    top20["correct"] == False
].copy()

wrong_top20[
    [
        "content_hash_id",
        "model_score",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "word_count",
        "content_age_days",
        "march_ctr",
        "future_opportunity"
    ]
]

,content_hash_id,model_score,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_age_days,march_ctr,future_opportunity
54440,content_cf445d75a938ea9a,0.913005,166.0,0.0,18.897590,4912,135,0.011834,False
11971,content_fbb1339100bd5e5b,0.897817,101.0,0.0,18.584158,3079,337,0.007874,False
30482,content_fa3d03c4de7ca6c6,0.894839,33.0,0.0,19.484848,1541,170,0.016393,False
52483,content_4dfb03531740366c,0.894047,95.0,0.0,12.094737,1768,213,0.005464,False
67956,content_fc7112e79d95962d,0.890287,4.0,0.0,3.500000,1694,157,0.003086,False


In [21]:
# Leakage audit

print("Features used by the model:")
print(FEATURES)

print("\nFuture/outcome columns in the test frame:")
future_columns = [
    c for c in test_model.columns
    if any(word in c.lower() for word in [
        "march",
        "future",
        "opportunity"
    ])
]

print(future_columns)

print("\nModel feature columns are clean:")
print(
    all(
        c not in [
            "march_ctr",
            "march_clicks",
            "march_impressions",
            "future_ctr",
            "future_opportunity"
        ]
        for c in FEATURES
    )
)

Features used by the model:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'word_count', 'content_age_days']

Future/outcome columns in the test frame:
['march_impressions', 'march_clicks', 'march_ctr', 'future_ctr_rank', 'future_opportunity']

Model feature columns are clean:
True


## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
